In [ ]:
#Vấn đề 1: Tiến hành tải dữ liệu vào chương trình ứng dụng Python và giải quyết vấn đề
#“Missing header in the csv file”
import pandas as pd

# Định nghĩa danh sách tiêu đề
column_names = ["Id", "Name", "Age", "Weight", "m0006", "m0612", "m1218", "f0006", "f0612", "f1218"]
# Tải dữ liệu và ép Pandas không nhận dòng đầu làm header, đồng thời gán danh sách tên cột
df = pd.read_csv('patient_heart_rate.csv', names=column_names)
print(df.head())

In [ ]:
#Vấn đề 2: Xử lý vấn đề một cột lưu hỗn hợp nhiều dữ liệu, ở đây là cột “Name” chứa bao
#gồm “Firstname” và “Lastname”, giải pháp là ta sẽ tách ra làm 2 cột 
df[['Firstname', 'Lastname']] = df['Name'].str.split(expand=True)
df = df.drop('Name', axis=1)
print(df)

In [ ]:
#Vấn đề 3: Cột Weight có vấn đề về không thống nhất các đơn vị đo lường trong dữ liệu.
#Ta sẽ chuyển các đơn vị về thành đơn vị chuẩn “kg”
weight = df["weight"]
for i in range (0, len(weight)):
          x = str(weight[i])
          if "lbs" in x[-3:]:
                  x = x [:-3]
                  float_x = float(x)
                  y = int(float_x/2.2)
                  y = str(y) + "kgs"
                  weight[i] = y
print(df)

In [ ]:
#Vấn đề 4: Vấn đề về xuất hiện dòng dữ liệu rỗng (không có giá trị: NaN). Giải pháp có 
#thể đưa ra là xóa bỏ
df.dropna(how = "all", inplace=True)
print(df)

In [ ]:
#Vấn đề 5: Có nhiều dòng dữ liệu bị trùng lắp thông tin hoàn toàn[fullname, lastname,
#age, weight,....], giải pháp đưa ra là chỉ giữ lại một dòng dữ liệu, tuy nhiên giải pháp phải
#dựa trên nghiệp vụ của tập dữ liệu và quan sát của người xử lý.
df = df.drop_duplicates(subset= ["Firstname", "Lastname", "Age", "Weight"])
print(df)

In [ ]:
#Vấn đề 6: Xuất hiện dữ liệu bị ảnh hưởng bởi lỗi non-ASCII, không định dạng ASCII.
#Giải pháp: Tùy vào nghiệp vụ ta có thể: xóa dữ liệu tại đó, thay thế bằng dữ liệu khác
#hoặc thay bằng việc đánh dấu bằng một kí tự khác (ví dụ: ‘warning’)
df.Firstname.replace({r'[^\x00-\x7F]+':''}, regex=True, inplace=True)
df.Lastname.replace({r'[^\x00-\x7F]+':''}, regex=True, inplace=True)
print(df)

In [ ]:
# VẤN ĐỀ 7: XỬ LÝ GIÁ TRỊ BỊ MẤT (MISSING VALUES)

df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

print("--- THỐNG KÊ SỐ LƯỢNG DỮ LIỆU THIẾU BAN ĐẦU ---")
print(df[['Age', 'Weight']].isnull().sum())
print("-" * 50)

df = df.dropna(subset=['Age', 'Weight'], how='all')

age_mean = df['Age'].mean()
print(f"=> Giá trị trung bình (Mean) của cột Age dùng để thay thế là: {age_mean:.2f}")
print("-" * 50)

df['Age'] = df['Age'].fillna(age_mean)

print(df)

In [ ]:
#Vấn đề 8: “một cột chứa quá nhiều thông tin cần được phân rã”, như trong bài toán này ta
#thấy header “m0006” chứa các nội dung bao gồm: m → male, 1218 ~ 12-18 (mm-dd).
#Còn giá trị thì là kết quả huyết áp.
df = pd.nelt(df, id_vars = ["Id", "Age", "Weight", "Firstname", "Lastname"], value_name = "pulseRate", var_name="sex_and_time").sort_values(["Id", "Age", "Weight", "Firstname", "Lastname"])
tmp_df = df["sex_and_time"].str.extract("(\D)(\d+)(\d{2}", expand=True)
tmp_df.colums = ["Sex", "hours_lower", "hours_upper"]
tmp_df[Time] = tmp_df["hours_lower"] + "-" + tmp_df["hours_upper"]
df = pd.concat([df, tmp_df], axis=1)
df = df.drop(["sex_and_time", "hours_lower", "hours_upper"], axis = 1)
df = df.dropna()
df.to_csv("outputcleanup,csv", index = False)
print(df)

In [ ]:


import numpy as np
import pandas as pd

pulse_cols = ["m0006", "m0612", "m1218", "f0006", "f0612", "f1218"]

for col in pulse_cols:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace("-", ""), errors="coerce")

df["Sex"] = df["Name"].apply(
    lambda x: "F" if "Mini" in str(x) or "Brianna" in str(x) else "M"
)

print("--- TỈ LỆ DỮ LIỆU THIẾU TRÊN CÁC CỘT HUYẾT ÁP ---")
total_rows = len(df)
for col in pulse_cols:
    missing_count = df[col].isnull().sum()
    missing_rate = (missing_count / total_rows) * 100
    print(f"Cột {col}: Thiếu {missing_count}/{total_rows} dòng ({missing_rate:.2f}%)")
print("-" * 60)


def impute_row(row, df_current):
    vals = [row[c] for c in pulse_cols]

    for i in range(len(vals)):
        if np.isnan(vals[i]):
            prev1 = vals[i - 1] if i - 1 >= 0 else np.nan
            prev2 = vals[i - 2] if i - 2 >= 0 else np.nan
            next1 = vals[i + 1] if i + 1 < len(vals) else np.nan
            next2 = vals[i + 2] if i + 2 < len(vals) else np.nan

            if not np.isnan(prev1) and not np.isnan(next1):
                vals[i] = (prev1 + next1) / 2
                continue

            if not np.isnan(prev1) and not np.isnan(prev2):
                vals[i] = (prev1 + prev2) / 2
                continue

            if not np.isnan(next1) and not np.isnan(next2):
                vals[i] = (next1 + next2) / 2
                continue

            valid_person_vals = [v for v in vals if not np.isnan(v)]
            if len(valid_person_vals) > 0:
                vals[i] = np.mean(valid_person_vals)
                continue

            gender_mean = df_current[df_current["Sex"] == row["Sex"]][
                pulse_cols[i]
                ].mean()
            if not np.isnan(gender_mean):
                vals[i] = gender_mean
                continue

            global_mean = df_current[pulse_cols[i]].mean()
            if not np.isnan(global_mean):
                vals[i] = global_mean
            else:
                vals[i] = 72.0 
    for idx, col in enumerate(pulse_cols):
        row[col] = vals[idx]
    return row


df = df.apply(lambda row: impute_row(row, df), axis=1)

df = df.drop(columns=["Sex"])

print("--- KẾT QUẢ BẢNG DỮ LIỆU SAU KHI ĐIỀN KHUYẾT HUYẾT ÁP ---")
print(df[pulse_cols].head(10))

In [ ]:
# VẤN ĐỀ 12: RÚT GỌN, REINDEX VÀ LƯU TRỮ DỮ LIỆU SẠCH

pulse_cols = ["m0006", "m0612", "m1218", "f0006", "f0612", "f1218"]
df_clean = df.dropna(subset=pulse_cols, how='all')

df_clean = df_clean.dropna(subset=['Name', 'Age', 'Weight'], how='all')

df_clean = df_clean.reset_index(drop=True)

df_clean.to_csv('patient_heart_rate_clean.csv', index=False, encoding='utf-8')

print("--- BẢNG DỮ LIỆU SAU KHI RÚT GỌN VÀ REINDEX THÀNH CÔNG ---")
print(df_clean)
print("\n=> Đã lưu file sạch thành công với tên: 'patient_heart_rate_clean.csv'")